# Mejora 6 - Ensamble por voto suave

**Proyecto - Parte 1: Machine Learning clasico**

**Estudiantes:**

[Nombre integrante 1] - [codigo]

[Nombre integrante 2] - [codigo]

**Curso:** ISIS2611 - Aprendizaje de Maquina

---

## Contexto

La comparacion justa de la Mejora 4 (seccion 8, 15 particiones) mostro que los tres modelos que mejoran al modelo actual -el apilamiento, el modelo en dos etapas (Mejora 1) y el modelo en dos etapas + SVM (Mejora 4)- quedan **empatados entre si**: ninguna diferencia entre ellos supera el ruido. El notebook principal y la Mejora 3 tambien muestran que la regresion logistica con ajuste fino de `C` esta en esa misma zona. Al mismo tiempo, el README del proyecto registra que estos modelos **no predicen lo mismo**: cambian entre 112 y 127 de las 3.000 predicciones de `eval.csv` entre si.

Esa combinacion -desempeno empatado, predicciones distintas- es exactamente la situacion en la que un **ensamble por voto suave** (promediar las probabilidades de varios modelos y quedarse con la clase de mayor probabilidad promedio) suele ayudar: si los errores de cada modelo no son los mismos casos, promediar probabilidades reduce la varianza sin necesitar un modelo nuevo ni una busqueda de hiperparametros adicional. Es la misma idea que ya usa el apilamiento (`StackingClassifier`) de la Exploracion de modelos alternativos, solo que en vez de que un metamodelo aprenda a combinar las salidas, aqui se promedian directamente con el mismo peso.

## Requisitos de la guia

| Requisito | Como se cumple |
|:---|:---|
| No usar aprendizaje profundo, transformers ni embeddings preentrenados. | No se usa ninguno. El ensamble combina los mismos modelos de scikit-learn ya usados en notebooks anteriores. |
| Usar unicamente modelos de scikit-learn. | Se usa `VotingClassifier` de scikit-learn para combinar los componentes (regresion logistica, Naive Bayes, SVM lineal y los modelos en dos etapas, estos ultimos compatibles con la API de scikit-learn). |
| Resultados replicables. | Semillas fijas, versiones en `requirements.txt` y todo el flujo en este notebook (seccion 9). |

## 1. Importación de librerías

In [1]:
import os
import re
import time
import unicodedata
import warnings
from collections import Counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from nltk.stem import SnowballStemmer

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

RANDOM_STATE = 42
ORDEN_CLASES = ['negativo', 'neutral', 'positivo']

# Rutas del proyecto (este notebook está en la carpeta notebooks/)
RUTA_DATOS = os.path.join('..', 'data')
RUTA_MODELOS = os.path.join('..', 'models')
RUTA_ENVIOS = os.path.join('..', 'submissions')
NOMBRE_MODELO = 'ensamble_voto_suave'

os.makedirs(RUTA_MODELOS, exist_ok=True)
os.makedirs(RUTA_ENVIOS, exist_ok=True)

## 2. Datos y partición

In [2]:
train = pd.read_csv(os.path.join(RUTA_DATOS, 'train.csv'))
evaluacion = pd.read_csv(os.path.join(RUTA_DATOS, 'eval.csv'))
muestra_envio = pd.read_csv(os.path.join(RUTA_DATOS, 'sample_submission.csv'))

print('train:', train.shape, '| eval:', evaluacion.shape, '| sample_submission:', muestra_envio.shape)

train: (12000, 3) | eval: (3000, 2) | sample_submission: (3000, 2)


In [3]:
X = train[['text']]
y = train['label']

# Misma partición 80/20 (semilla 42) de todos los notebooks anteriores
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

Entrenamiento: (9600, 1) | Prueba: (2400, 1)


## 3. Los cinco candidatos que se van a combinar

Se reutilizan, sin cambiar sus hiperparámetros, los cinco modelos que ya se entrenaron y evaluaron en notebooks anteriores:

| Candidato | Notebook de origen | Hiperparámetros |
|:---|:---|:---|
| `actual` | Notebook principal | Regresión logística L1, `C = 10`, representación por bloques |
| `ajuste_fino` | Mejora 3 | Regresión logística L1, `C = 7` (mejor envío público hasta ahora, 0,90666) |
| `apilamiento` | Exploración de modelos alternativos | `StackingClassifier` (regresión logística + Naive Bayes + SVM lineal, metamodelo de regresión logística) |
| `dos_etapas` | Mejora 1 | `ModeloDosEtapas` sin SVM, `C_etapa2 = 0,1` |
| `dos_etapas_svm` | Mejora 4 | `ModeloDosEtapas` con SVM, `C_etapa2 = 0,03` |

No se cargan desde `models/*.joblib` porque esos archivos guardan el pipeline completo y referencian funciones que viven en cada notebook (ver README, "Usar el modelo guardado fuera del notebook"); en vez de eso se reconstruyen aquí con el mismo código y las mismas semillas de sus notebooks de origen, así que sus métricas individuales deben coincidir exactamente con las ya reportadas.

In [4]:
stemmer = SnowballStemmer('spanish')
cache_raices = {}


def raiz(palabra):
    if palabra not in cache_raices:
        cache_raices[palabra] = stemmer.stem(palabra)
    return cache_raices[palabra]


def quitar_tildes(texto):
    texto = unicodedata.normalize('NFKD', texto)
    return ''.join(c for c in texto if not unicodedata.combining(c))


def limpiar_texto(texto, stop=None, stemming=True):
    texto = quitar_tildes(texto.lower())
    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\d+', ' ', texto)
    tokens = texto.split()
    if stop:
        tokens = [t for t in tokens if t not in stop]
    if stemming:
        tokens = [raiz(t) for t in tokens]
    return ' '.join(tokens)


def separar_oraciones(texto):
    return [o.strip() for o in re.split(r'(?<=[.!?])\s+', texto) if o.strip()]


# Conectores que introducen la opinión que se impone sobre la anterior
CONECTORES = r'\b(?:pero|aunque|sin embargo|aun asi|con todo|a fin de cuentas|al final)\b'


def extraer_completo(textos, stemming=True):
    return [limpiar_texto(t, stemming=stemming) for t in textos]


def extraer_ultima(textos, stemming=True):
    return [limpiar_texto(separar_oraciones(t)[-1], stemming=stemming) for t in textos]


def extraer_clausula_final(textos, stemming=True):
    salida = []
    for t in textos:
        ultima = quitar_tildes(separar_oraciones(t)[-1].lower())
        salida.append(limpiar_texto(re.split(CONECTORES, ultima)[-1], stemming=stemming))
    return salida


def bloque_tfidf(nombre, extractor, rango=(1, 2)):
    return (nombre,
            Pipeline([('extraer', FunctionTransformer(extractor)),
                      ('tfidf', TfidfVectorizer(ngram_range=rango))]),
            'text')


def construir_representacion(bloques=('completo', 'ultima', 'clausula'), rango=(1, 2)):
    extractores = {'completo': extraer_completo, 'ultima': extraer_ultima, 'clausula': extraer_clausula_final}
    return ColumnTransformer([bloque_tfidf(b, extractores[b], rango) for b in bloques])


def metricas(y_real, y_pred):
    return {'Exactitud': accuracy_score(y_real, y_pred),
            'Precisión (macro)': precision_score(y_real, y_pred, average='macro'),
            'Recall (macro)': recall_score(y_real, y_pred, average='macro'),
            'F1 (macro)': f1_score(y_real, y_pred, average='macro')}

In [5]:
# Funciones del modelo en dos etapas (Mejoras 1 y 4), sin cambios
def separar_trozos(texto):
    """Parte cada oración en trozos usando los conectores de contraste."""
    trozos = []
    for oracion in separar_oraciones(texto):
        partes = re.split(CONECTORES, quitar_tildes(oracion.lower()))
        for i, parte in enumerate(partes):
            if parte.strip():
                trozos.append((parte.strip(), int(i > 0)))
    return trozos


def entrenar_etapa1(textos, etiquetas, C):
    trozos, etiquetas_trozos = [], []
    for texto, etiqueta in zip(textos, etiquetas):
        for trozo, _ in separar_trozos(texto):
            trozos.append(limpiar_texto(trozo))
            etiquetas_trozos.append(etiqueta)
    vectorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    clasificador = LogisticRegression(C=C, max_iter=2000)
    clasificador.fit(vectorizador.fit_transform(trozos), etiquetas_trozos)
    return vectorizador, clasificador


K_ULTIMOS = 4
N_INICIOS = 12


def comienzo_oracion(oracion):
    palabras = re.sub(r'[^\w\s]', '', quitar_tildes(oracion.lower())).split()
    return ' '.join(palabras[:2])


def comienzos_frecuentes(textos, n=N_INICIOS):
    conteo = Counter(comienzo_oracion(o) for t in textos for o in separar_oraciones(t)[1:])
    return [c for c, _ in conteo.most_common(n)]


class ModeloDosEtapas(ClassifierMixin, BaseEstimator):
    """Clasificador de reseñas en dos etapas (igual a la Mejora 4, sin cambios)."""

    def __init__(self, C_etapa1=3, C_etapa2=0.1, C_svm=0.3, usar_svm=True, n_particiones=5, random_state=42):
        self.C_etapa1 = C_etapa1
        self.C_etapa2 = C_etapa2
        self.C_svm = C_svm
        self.usar_svm = usar_svm
        self.n_particiones = n_particiones
        self.random_state = random_state

    def _entrenar_componentes(self, textos, etiquetas):
        datos = pd.DataFrame({'text': textos.values})
        representacion = construir_representacion().fit(datos)
        matriz = representacion.transform(datos)
        comp = {'representacion': representacion,
                'regresion': LogisticRegression(C=10, penalty='l1', solver='liblinear', max_iter=3000,
                                                random_state=RANDOM_STATE).fit(matriz, etiquetas),
                'bayes': MultinomialNB(alpha=0.3).fit(matriz, etiquetas)}
        if self.usar_svm:
            comp['svm'] = LinearSVC(C=self.C_svm, random_state=RANDOM_STATE).fit(matriz, etiquetas)
        comp['vectorizador'], comp['clasificador'] = entrenar_etapa1(textos, etiquetas, self.C_etapa1)
        return comp

    def _caracteristicas(self, textos, comp):
        datos = pd.DataFrame({'text': textos.values})
        matriz = comp['representacion'].transform(datos)
        prob_regresion = comp['regresion'].predict_proba(matriz)
        prob_bayes = comp['bayes'].predict_proba(matriz)

        trozos_por_texto = [separar_trozos(t) for t in textos]
        oraciones_por_texto = [separar_oraciones(t) for t in textos]
        probs_trozos = comp['clasificador'].predict_proba(comp['vectorizador'].transform(
            [limpiar_texto(trozo) for lista in trozos_por_texto for trozo, _ in lista]))
        probs_oraciones = comp['clasificador'].predict_proba(comp['vectorizador'].transform(
            [limpiar_texto(o) for lista in oraciones_por_texto for o in lista]))

        filas = []
        ini_t = ini_o = 0
        for k in range(len(textos)):
            trozos, oraciones = trozos_por_texto[k], oraciones_por_texto[k]
            P = probs_trozos[ini_t:ini_t + len(trozos)]
            Po = probs_oraciones[ini_o:ini_o + len(oraciones)]
            ini_t += len(trozos)
            ini_o += len(oraciones)

            fila = list(prob_regresion[k])
            for j in range(1, K_ULTIMOS + 1):
                if j <= len(trozos):
                    fila += list(P[-j]) + [trozos[-j][1]]
                else:
                    fila += [0, 0, 0, 0]

            polaridad = P[:, 2] - P[:, 0]
            con_opinion = np.where(P[:, 1] < 0.5)[0]
            fila += [polaridad[con_opinion[-1]] if len(con_opinion) > 0 else 0,
                     polaridad[con_opinion[-2]] if len(con_opinion) > 1 else 0,
                     polaridad[con_opinion[0]] if len(con_opinion) > 0 else 0,
                     len(con_opinion), P[:, 2].max(), P[:, 0].max(), P[:, 1].mean(), len(trozos)]

            fila += list(prob_bayes[k])
            oraciones_con_opinion = np.where(Po[:, 1] < 0.5)[0]
            for pos in [-1, -2]:
                comienzo = ''
                if len(oraciones_con_opinion) >= -pos:
                    comienzo = comienzo_oracion(oraciones[oraciones_con_opinion[pos]])
                fila += [int(comienzo == c) for c in self.comienzos_]
            filas.append(fila)

        caracteristicas = np.array(filas)
        if self.usar_svm:
            caracteristicas = np.hstack([caracteristicas, comp['svm'].decision_function(matriz)])
        return caracteristicas

    def fit(self, X, y):
        textos = pd.Series(np.asarray(X['text']))
        etiquetas = pd.Series(np.asarray(y))
        self.classes_ = np.array(sorted(etiquetas.unique()))
        self.comienzos_ = comienzos_frecuentes(textos)

        particiones = StratifiedKFold(n_splits=self.n_particiones, shuffle=True, random_state=self.random_state)
        caracteristicas = None
        for idx_ent, idx_val in particiones.split(textos, etiquetas):
            comp = self._entrenar_componentes(textos.iloc[idx_ent], etiquetas.iloc[idx_ent])
            fila_val = self._caracteristicas(textos.iloc[idx_val], comp)
            if caracteristicas is None:
                caracteristicas = np.zeros((len(textos), fila_val.shape[1]))
            caracteristicas[idx_val] = fila_val

        self.etapa2_ = Pipeline([('escala', StandardScaler()),
                                 ('modelo', LogisticRegression(C=self.C_etapa2, max_iter=3000))])
        self.etapa2_.fit(caracteristicas, etiquetas)

        self.componentes_ = self._entrenar_componentes(textos, etiquetas)
        return self

    def predict_proba(self, X):
        textos = pd.Series(np.asarray(X['text']))
        return self.etapa2_.predict_proba(self._caracteristicas(textos, self.componentes_))

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

In [6]:
def lr_l1(C=10):
    return LogisticRegression(C=C, penalty='l1', solver='liblinear', max_iter=3000, random_state=RANDOM_STATE)


def construir_candidatos():
    return {
        'actual': Pipeline([('rep', construir_representacion()), ('modelo', lr_l1(10))]),
        'ajuste_fino': Pipeline([('rep', construir_representacion()), ('modelo', lr_l1(7))]),
        'apilamiento': Pipeline([('rep', construir_representacion()),
                                 ('modelo', StackingClassifier(
                                     [('lr', lr_l1()), ('nb', MultinomialNB(alpha=0.5)),
                                      ('svm', LinearSVC(C=0.3, random_state=RANDOM_STATE))],
                                     final_estimator=LogisticRegression(max_iter=3000, random_state=RANDOM_STATE), cv=5))]),
        'dos_etapas': ModeloDosEtapas(usar_svm=False, C_etapa2=0.1),
        'dos_etapas_svm': ModeloDosEtapas(usar_svm=True, C_etapa2=0.03),
    }


candidatos = construir_candidatos()
filas_individuales = {}
predicciones_individuales = {}
t0 = time.time()
for nombre, modelo in candidatos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    predicciones_individuales[nombre] = pred
    filas_individuales[nombre] = metricas(y_test, pred)
print(f'Los cinco candidatos se entrenaron en {time.time() - t0:.0f} segundos')
pd.DataFrame(filas_individuales).T.round(4)

Los cinco candidatos se entrenaron en 618 segundos


,Exactitud,Precisión (macro),Recall (macro),F1 (macro)
actual,0.8946,0.8987,0.8998,0.8993
ajuste_fino,0.8975,0.9013,0.9026,0.9019
apilamiento,0.9000,0.9045,0.9049,0.9047
dos_etapas,0.8967,0.9017,0.9017,0.9017
dos_etapas_svm,0.8988,0.9037,0.9037,0.9036


Las métricas de cada candidato por separado coinciden (hasta el redondeo) con las ya reportadas en sus notebooks de origen: es la misma partición, las mismas semillas y el mismo código.

## 4. El ensamble: `VotingClassifier` con voto suave

`VotingClassifier(voting='soft')` promedia la probabilidad que cada candidato le asigna a cada clase y se queda con la de mayor promedio. Como ningún candidato domina claramente a los demás (todos están dentro del ruido entre sí, según la Mejora 4) y todos ya fueron validados en su propio notebook, se combinan **los cinco** con el mismo peso, sin elegir un subconjunto a partir de qué tan bien les fue en este conjunto de prueba -eso sería repetir el mismo error que ya identificó la Mejora 3 al elegir la mejor de varias configuraciones casi iguales.

In [7]:
ensamble = VotingClassifier(
    estimators=[(nombre, modelo) for nombre, modelo in candidatos.items()],
    voting='soft',
)
ensamble.fit(X_train, y_train)
pred_ensamble = ensamble.predict(X_test)

tabla_ensamble = pd.DataFrame({**filas_individuales, 'Ensamble (voto suave)': metricas(y_test, pred_ensamble)}).T
tabla_ensamble.round(4).sort_values('F1 (macro)', ascending=False)

,Exactitud,Precisión (macro),Recall (macro),F1 (macro)
Ensamble (voto suave),0.9004,0.9048,0.9054,0.9051
apilamiento,0.9000,0.9045,0.9049,0.9047
dos_etapas_svm,0.8988,0.9037,0.9037,0.9036
ajuste_fino,0.8975,0.9013,0.9026,0.9019
dos_etapas,0.8967,0.9017,0.9017,0.9017
actual,0.8946,0.8987,0.8998,0.8993


En este conjunto de prueba el ensamble queda primero en exactitud y F1-macro, aunque muy cerca del apilamiento solo (que ya es, en sí mismo, una forma de combinar modelos). La pregunta, como siempre en este proyecto, es si esa ventaja es real o es ruido de esta partición de 2.400 reseñas.

## 5. ¿La diferencia es real? Prueba de bootstrap pareado

Las Mejoras 2, 3, 4 y 5 responden esta pregunta reentrenando cada modelo sobre 15 particiones distintas. Aquí no es práctico: los dos modelos en dos etapas tardan unos 5 minutos **cada uno** en entrenarse una sola vez (reentrenan sus componentes internos con su propia validación cruzada de 5 particiones), así que 15 repeticiones de los cinco candidatos tomarían varias horas.

En su lugar se usa una alternativa estándar para este caso -la **prueba de bootstrap pareado** (Efron & Tibshirani, 1993; es la misma idea que usan Berg-Kirkpatrick et al., 2012, para comparar sistemas de PLN sin reentrenarlos)-: en vez de generar nuevas particiones de entrenamiento, se remuestrean con reemplazo las 2.400 reseñas **del conjunto de prueba ya predicho** (ninguna de las dos predicciones que se comparan cambia; solo cambia qué reseñas se promedian) y se mide, en miles de remuestreos, qué tan seguido el ensamble tiene mayor exactitud que el rival. Es menos informativo que reentrenar con particiones nuevas (no dice nada sobre la varianza del entrenamiento, solo sobre la incertidumbre de medir con 2.400 reseñas), pero es computacionalmente viable y responde la misma pregunta de fondo: ¿la diferencia observada podría deberse solo a qué reseñas cayeron en el conjunto de prueba?

In [8]:
rng = np.random.RandomState(RANDOM_STATE)
B = 5000
n_prueba = len(y_test)
y_test_arr = np.asarray(y_test)
idx_base = np.arange(n_prueba)


def bootstrap_pareado(pred_a, pred_b, etiquetas=y_test_arr, repeticiones=B):
    aciertos_a = (pred_a == etiquetas)
    aciertos_b = (pred_b == etiquetas)
    diferencias = np.empty(repeticiones)
    for i in range(repeticiones):
        idx = rng.choice(idx_base, size=n_prueba, replace=True)
        diferencias[i] = aciertos_a[idx].mean() - aciertos_b[idx].mean()
    return diferencias


filas_bootstrap = []
for rival in ['actual', 'ajuste_fino', 'apilamiento', 'dos_etapas', 'dos_etapas_svm']:
    diferencias = bootstrap_pareado(pred_ensamble, predicciones_individuales[rival])
    filas_bootstrap.append({
        'Rival': rival,
        'Diferencia media de exactitud': diferencias.mean(),
        '% de remuestreos donde el ensamble gana': (diferencias > 0).mean() * 100,
    })

tabla_bootstrap = pd.DataFrame(filas_bootstrap).set_index('Rival')
tabla_bootstrap.round(4)

,Diferencia media de exactitud,% de remuestreos donde el ensamble gana
Rival,,
actual,0.0059,97.90
ajuste_fino,0.0029,84.22
apilamiento,0.0005,54.54
dos_etapas,0.0037,92.28
dos_etapas_svm,0.0017,74.14


- **Frente al modelo actual, la mejora es clara:** el ensamble gana en el 97-98 % de los remuestreos, el mismo nivel de evidencia que la Mejora 4 encontró para sus mejores modelos individuales.
- **Frente al ajuste fino (el mejor envío público hasta ahora), hay un indicio de mejora pero no es concluyente:** el ensamble gana en torno al 84 % de los remuestreos. Para la vara que usa este proyecto (equivalente a necesitar una victoria muy consistente, como el 12 de 15 de las Mejoras 3 y 5) esto queda corto de ser una mejora demostrada.
- **Frente al apilamiento, es un empate casi perfecto** (alrededor de 50 %): no sorprende, porque el apilamiento ya es, por su cuenta, una forma de combinar modelos con información parecida.

## 6. ¿Dónde ayuda el ensamble?

In [9]:
errores_actual = predicciones_individuales['actual'] != y_test_arr
errores_ensamble = pred_ensamble != y_test_arr

tabla_errores = pd.DataFrame({
    'Error solo en el modelo actual (el ensamble corrige)': (errores_actual & ~errores_ensamble).sum(),
    'Error solo en el ensamble (el modelo actual acertaba)': (~errores_actual & errores_ensamble).sum(),
    'Error en ambos': (errores_actual & errores_ensamble).sum(),
}, index=['Reseñas']).T
tabla_errores

,Reseñas
Error solo en el modelo actual (el ensamble corrige),31
Error solo en el ensamble (el modelo actual acertaba),17
Error en ambos,222


El ensamble corrige bastantes más reseñas de las que daña respecto al modelo actual, lo que es consistente con el resultado de la sección anterior (mejora clara frente al modelo actual). Esta es la lógica detrás del voto suave: cuando los cinco candidatos no fallan en las mismas reseñas, promediar sus probabilidades favorece a la clase que la mayoría ve con más confianza, incluso si ningún candidato por separado es claramente el mejor.

## 7. Modelo final y predicciones para la competencia

El ensamble no mejora de forma concluyente al ajuste fino (sección 5), así que **no reemplaza** a `regresion_logistica_ajuste_fino` como mejor envío del grupo. Aun así, es una técnica distinta de las ya probadas (ninguno de los otros envíos combina modelos por votación) y mejora de forma clara al modelo actual, así que se entrena con las 12.000 reseñas etiquetadas y se sube como uno de los envíos adicionales que pide la guía.

In [10]:
modelo_final = VotingClassifier(
    estimators=[(nombre, modelo) for nombre, modelo in construir_candidatos().items()],
    voting='soft',
)
t0 = time.time()
modelo_final.fit(X, y)
print(f'Entrenado con los datos completos en {time.time() - t0:.0f} segundos')

pred_eval = modelo_final.predict(evaluacion[['text']])
envio = pd.DataFrame({'id': evaluacion['id'], 'answer': pred_eval})
envio.to_csv(os.path.join(RUTA_ENVIOS, f'submission_{NOMBRE_MODELO}.csv'), index=False)

print('Mismas columnas que sample_submission:', list(envio.columns) == list(muestra_envio.columns))
print('Mismos ids y en el mismo orden:', (envio['id'] == muestra_envio['id']).all())

otros_envios = {
    'Modelo actual (regresion_logistica_bloques)': 'submission_regresion_logistica_bloques.csv',
    'Mejor envío público (regresion_logistica_ajuste_fino)': 'submission_regresion_logistica_ajuste_fino.csv',
    'Apilamiento (exploración)': 'submission_exploracion_apilamiento.csv',
}
for nombre, archivo in otros_envios.items():
    otro = pd.read_csv(os.path.join(RUTA_ENVIOS, archivo))
    distintas = (envio['answer'] != otro['answer']).sum()
    print(f'Predicciones distintas respecto a {nombre}: {distintas} de {len(envio)}')

display(pd.DataFrame({'train (real)': train['label'].value_counts(normalize=True),
                      'eval (predicho)': envio['answer'].value_counts(normalize=True)}).reindex(ORDEN_CLASES).round(3))

Entrenado con los datos completos en 777 segundos


Mismas columnas que sample_submission: True
Mismos ids y en el mismo orden: True
Predicciones distintas respecto a Modelo actual (regresion_logistica_bloques): 72 de 3000
Predicciones distintas respecto a Mejor envío público (regresion_logistica_ajuste_fino): 64 de 3000
Predicciones distintas respecto a Apilamiento (exploración): 54 de 3000


,train (real),eval (predicho)
negativo,0.351,0.355
neutral,0.298,0.279
positivo,0.351,0.366


In [11]:
ruta_modelo = os.path.join(RUTA_MODELOS, f'{NOMBRE_MODELO}.joblib')
joblib.dump(modelo_final, ruta_modelo)

modelo_cargado = joblib.load(ruta_modelo)
print('El modelo guardado reproduce exactamente las mismas predicciones:',
      (modelo_cargado.predict(evaluacion[['text']]) == pred_eval).all())

El modelo guardado reproduce exactamente las mismas predicciones: True


## 8. Conclusiones

- **El ensamble por voto suave mejora de forma clara al modelo actual** (gana en 97-98 % de los remuestreos bootstrap), con la misma fuerza que los mejores modelos individuales de las Mejoras 1 y 4.
- **Frente al mejor envío público (`regresion_logistica_ajuste_fino`), la mejora es solo un indicio** (gana en torno al 84 % de los remuestreos): no alcanza el nivel de evidencia que este proyecto exige para declarar una mejora. Por eso el ajuste fino sigue siendo el envío recomendado del grupo.
- **Frente al apilamiento solo, el ensamble es un empate**: combinar cinco modelos no aporta mucho más que la combinación que ya hace el apilamiento con tres, porque varios de los candidatos comparten la misma representación por bloques y se equivocan en reseñas parecidas.
- **La lección general:** promediar modelos decorrelacionados sí reduce el error frente a cualquiera de ellos por separado (como lo muestra la comparación contra el modelo actual), pero el beneficio depende de qué tan distintos sean los errores de los modelos que se combinan. Dos modelos que ya son, en el fondo, formas de combinar las mismas piezas (bolsa de palabras + regresión logística + Naive Bayes + SVM) no se comportan como si fueran independientes.
- **Para Kaggle,** este envío aporta uno de los 5 envíos distintos que pide la guía con una técnica nueva (combinación de modelos, no una variación de hiperparámetros), y sirve como punto de comparación adicional para el percentil final de la competencia.

## 9. Uso de herramientas de IA generativa

**Declaración del uso.** Se utilizó Claude Code (modelo Claude Sonnet 5, de Anthropic) como apoyo para: proponer el ensamble por voto suave como oportunidad de mejora de menor riesgo (reutiliza modelos ya validados en vez de entrenar uno nuevo), generar un primer esqueleto del código de este notebook reutilizando las funciones y clases ya definidas en los notebooks de Mejoras 1 y 4, proponer la prueba de bootstrap pareado como alternativa computacionalmente viable a la comparación de 15 particiones (que hubiera tomado varias horas con los modelos en dos etapas), y redactar las explicaciones.

**Prompts utilizados.**

1. *"Tengo este proyecto de machine learning. El mejor modelo que hemos obtenido ha tenido un score en kaggle de 0.90666 Cómo podemos mejorar?"*, tras lo cual la herramienta revisó el PDF de la guía y los notebooks existentes y propuso, entre otras, un ensamble por voto suave de los modelos ya entrenados (de menor riesgo porque no requiere entrenar un modelo nuevo) y el marcado de negación (Mejora 5).
2. *"haz el ensamble por voto suave"*, para implementar esta propuesta.

**Análisis crítico del resultado.**

[Completar por el equipo: ¿qué partes del código y de las explicaciones generadas fueron correctas y útiles? ¿qué errores, imprecisiones o limitaciones se identificaron? ¿qué decisiones técnicas se cambiaron respecto a lo que propuso la herramienta y por qué? ¿qué conceptos del curso permitieron evaluar o corregir la respuesta?]

**Aportes propios del equipo.**

[Completar por el equipo: qué se desarrolló, modificó o decidió, qué ajustes se hicieron al código y a las explicaciones, y qué se aprendió del proceso.]

## 10. Cómo replicar los resultados

1. Crear un entorno con Python 3.9 e instalar las librerías con `pip install -r requirements.txt` (están en la raíz del proyecto).
2. Mantener la estructura de carpetas del proyecto: este notebook está en `notebooks/`, los datos en `data/`. El notebook se ejecuta desde la carpeta `notebooks/`.
3. Ejecutar todas las celdas en orden (`Restart & Run All`). Tarda unos 25 minutos: cada uno de los dos modelos en dos etapas se entrena dos veces (sobre el 80 % de entrenamiento y, al final, sobre los datos completos) y cada entrenamiento tarda unos 5 minutos.
4. Al terminar se generan `models/ensamble_voto_suave.joblib` y `submissions/submission_ensamble_voto_suave.csv`.

Todas las fuentes de aleatoriedad usan semillas fijas, por lo que las cifras se reproducen exactamente con las versiones de `requirements.txt`. Para cargar el modelo guardado fuera del notebook hay que ejecutar antes las celdas que definen las funciones y clases usadas por el pipeline (secciones 3).